# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

print("HF token loaded successfully.")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face successfully.")

HF token loaded successfully.
DuckDB connected to Hugging Face successfully.


In [2]:
print("""
UNIT OF ANALYSIS
One row represents one content item for one client on one report date.

TIME WINDOW
Feature window: 2026-02-01 to 2026-02-28
Label window: 2026-03-01 to 2026-03-31

The features use only information available during February.
The March window is used as the future outcome/label period.
""")


UNIT OF ANALYSIS
One row represents one content item for one client on one report date.

TIME WINDOW
Feature window: 2026-02-01 to 2026-02-28
Label window: 2026-03-01 to 2026-03-31

The features use only information available during February.
The March window is used as the future outcome/label period.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# Data contract: classify the fields used by the model

FEATURE_FIELDS = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position",
    "gsc_sum_position",
]

LABEL_FIELD = [
    "went_dark"
]

CONTEXT_FIELDS = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
]

EXCLUDED_FIELDS = {
    "client_hash_id": "Identifier only; used for grouping and joins, not as a predictive feature.",
    "content_hash_id": "Identifier only; used for grouping and joins, not as a predictive feature.",
    "report_date": "Date is used to define the feature/label windows rather than as a predictive feature.",
}

print("FEATURES:")
for x in FEATURE_FIELDS:
    print(" -", x)

print("\nLABEL:")
for x in LABEL_FIELD:
    print(" -", x)

print("\nCONTEXT:")
for x in CONTEXT_FIELDS:
    print(" -", x)

print("\nEXCLUDED:")
for k, v in EXCLUDED_FIELDS.items():
    print(f" - {k}: {v}")

FEATURES:
 - gsc_impressions
 - gsc_clicks
 - gsc_ctr
 - gsc_avg_position
 - gsc_sum_position

LABEL:
 - went_dark

CONTEXT:
 - client_hash_id
 - content_hash_id
 - report_date

EXCLUDED:
 - client_hash_id: Identifier only; used for grouping and joins, not as a predictive feature.
 - content_hash_id: Identifier only; used for grouping and joins, not as a predictive feature.
 - report_date: Date is used to define the feature/label windows rather than as a predictive feature.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [7]:
# Test the small sample file first

import duckdb

REL = "hf://datasets/FlyRank/internship-warehouse"

# Test the sample Parquet file
sample = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{REL}/fact_content_daily_performance_sample.parquet'
    )
    LIMIT 5
""").df()

sample

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance_sample.parquet' (HTTP 400)

In [8]:
print(con.sql("SELECT version()").fetchone())

('v1.3.2',)


In [9]:
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

print("httpfs loaded")

httpfs loaded


In [10]:
sample = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
    )
    LIMIT 5
""").df()

sample

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance_sample.parquet' (HTTP 400)